# Phase 1 - Continuous Pretraining (CPT) for Aya Expanse 8B on Tunisian Dialect

This notebook performs **continued pretraining** of `CohereLabs/aya-expanse-8b` on your cleaned Tunisian corpus from Hugging Face: `Syrinesmati/tunisian-dialect-corpus`.
Workflow:
1. Install and import dependencies
2. Load dataset and detect text column
3. Tokenize and pack text for causal language modeling
4. Configure LoRA/QLoRA training
5. Train and save checkpoints
6. Run a quick generation sanity check

In [1]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl huggingface_hub

In [2]:
import os
import math
import torch
from dataclasses import dataclass

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10


In [3]:
@dataclass
class CPTConfig:
    model_name: str = 'CohereLabs/aya-expanse-8b'
    dataset_name: str = 'Syrinesmati/tunisian-dialect-corpus'
    dataset_split: str = 'train'
    output_dir: str = '../../checkpoints/aya-expanse-8b-cpt-tunisian'
    max_seq_length: int = 1024
    eval_size: float = 0.01
    
    # Training
    learning_rate: float = 2e-4
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    warmup_ratio: float = 0.03
    logging_steps: int = 25
    save_steps: int = 200
    eval_steps: int = 200

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Quantization
    use_4bit: bool = True
    bnb_4bit_quant_type: str = 'nf4'
    bnb_4bit_compute_dtype: str = 'bfloat16'
    bnb_4bit_use_double_quant: bool = True

cfg = CPTConfig()
cfg

CPTConfig(model_name='CohereLabs/aya-expanse-8b', dataset_name='Syrinesmati/tunisian-dialect-corpus', dataset_split='train', output_dir='../../checkpoints/aya-expanse-8b-cpt-tunisian', max_seq_length=1024, eval_size=0.01, learning_rate=0.0002, num_train_epochs=1, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16, warmup_ratio=0.03, logging_steps=25, save_steps=200, eval_steps=200, lora_r=16, lora_alpha=32, lora_dropout=0.05, use_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype='bfloat16', bnb_4bit_use_double_quant=True)

## Optional: Hugging Face Login
Run only if your model/dataset access requires authentication.

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
dataset = load_dataset(cfg.dataset_name, split=cfg.dataset_split)
print(dataset)
print('Columns:', dataset.column_names)

README.md: 0.00B [00:00, ?B/s]

tunisian_arabic_clean.parquet:   0%|          | 0.00/208M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1180174 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'source'],
    num_rows: 1180174
})
Columns: ['text', 'source']


In [5]:
# The dataset's text column is named 'text' in our corpus.
# Use it directly rather than searching for column names.
text_col = 'text'
print('Using text column:', text_col)
# Filter out empty or null rows in the text column
dataset = dataset.filter(lambda x: x[text_col] is not None and str(x[text_col]).strip() != '')

Using text column: text


Filter:   0%|          | 0/1180174 [00:00<?, ? examples/s]

In [11]:
print('Dataset size after filtering:', len(dataset))

Dataset size after filtering: 1180174


## Load Aya (Your Base Snippet)
This cell follows your Aya loading pattern and then adapts it for training.

In [8]:
compute_dtype = torch.bfloat16 if cfg.bnb_4bit_compute_dtype == 'bfloat16' else torch.float16

bnb_config = None
if cfg.use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=cfg.bnb_4bit_use_double_quant,
    )

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype if not cfg.use_4bit else None,
    device_map='auto',
)

messages = [
    {'role': 'user', 'content': 'Who are you?'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

/home/ala/myenv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

/home/ala/myenv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


I am Coral, an AI-assistant chatbot built by Cohere. I am designed to engage in conversations with users, answer questions, provide explanations, and assist with a wide range of tasks. My


In [9]:
def tokenize_fn(batch):
    return tokenizer(batch[text_col], truncation=False)

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset.column_names,
    desc='Tokenizing dataset',
)

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples['input_ids'])
    total_length = (total_length // cfg.max_seq_length) * cfg.max_seq_length
    result = {
        k: [t[i : i + cfg.max_seq_length] for i in range(0, total_length, cfg.max_seq_length)]
        for k, t in concatenated_examples.items()
    }
    result['labels'] = result['input_ids'].copy()
    return result

lm_dataset = tokenized.map(group_texts, batched=True, desc='Packing into fixed blocks')
split = lm_dataset.train_test_split(test_size=cfg.eval_size, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(len(train_dataset))
print(len(eval_dataset))

Tokenizing dataset:   0%|          | 0/1180174 [00:00<?, ? examples/s]

Packing into fixed blocks:   0%|          | 0/1180174 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 81798
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 827
})


In [12]:
print(len(train_dataset))
print(len(eval_dataset))

81798
827


In [10]:
if cfg.use_4bit:
    model = prepare_model_for_kbit_training(model)

target_modules = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj'
]

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=target_modules,
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,069,976,064 || trainable%: 0.5197


In [ ]:
os.makedirs(cfg.output_dir, exist_ok=True)

# Calculate warmup_steps
total_train_samples = len(train_dataset)
batch_size = cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps
total_steps = (total_train_samples / batch_size) * cfg.num_train_epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)

training_args = TrainingArguments(
    f"{cfg.output_dir}",
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    warmup_steps=warmup_steps,
    eval_strategy='steps',
    eval_steps=cfg.eval_steps,
    save_steps=cfg.save_steps,
    logging_steps=cfg.logging_steps,
    save_total_limit=3,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    gradient_checkpointing=True,
    report_to='none',
    optim='paged_adamw_8bit' if cfg.use_4bit else 'adamw_torch',
    lr_scheduler_type='cosine',
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    )

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    )

trainer

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
trainer.train(resume_from_checkpoint=True)

In [ ]:
from peft import PeftModel

# Load base model (quantized config if used) and attach trained adapter
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype if not cfg.use_4bit else None,
    device_map='auto',
    )
model_with_adapter = PeftModel.from_pretrained(base_model, cfg.output_dir)

# Quick Tunisian prompt to sanity-check the adapter
messages = [
    {'role': 'user', 'content': 'اليوم الطقس مزيان شنوة نجم نعمل؟'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
    )
inputs = {k: v.to(model_with_adapter.device) for k, v in inputs.items()}
outputs = model_with_adapter.generate(**inputs, max_new_tokens=120, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

## Notes
- For full CPT without LoRA, use full-parameter training (requires very large GPU memory).
- For stability, start with 1 epoch and inspect validation loss before extending training.
- You can push adapter checkpoints to Hugging Face Hub using `trainer.push_to_hub()` after login.